# 07 Background Nuisance Feature Probe

역할: background feature가 foreground feature의 global appearance shift response를 얼마나 설명하는지 검증한다. 이 노트북은 anomaly detector를 학습하지 않는다. 목적은 background-derived nuisance calibration 가정이 방어 가능한지 확인하는 것이다.

## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. 실험 코드는 source tree와 data augmentation helper만 사용한다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )


colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git',
                '-C',
                str(colab_checkout),
                'checkout',
                'FETCH_HEAD',
                '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/configs',
                'experiments/validation/condition_shift_baseline/notebook/07_background_nuisance_probe.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for import_root in [SRC_ROOT, SRC_ROOT / 'orchestration']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Probe Settings And Data Readiness

`breakfast_box`의 clean normal 이미지와 global appearance shift 조건만 사용한다. background/foreground mask는 기존 clean grounding mask의 foreground union을 사용한다. foreground/background 분리가 틀리면 이 probe 자체가 흔들리므로, 여기서는 segmentation을 연구 기여로 주장하지 않는다.

In [ ]:
CATEGORY = 'breakfast_box'
SAMPLE_IDS = ['000.png', '001.png', '002.png', '003.png', '004.png']
SHIFT_SPECS = [
    ('brightness', 'high'),
    ('gaussian_blur', 'high'),
    ('gaussian_noise', 'high'),
    ('low_light', 'high'),
]
IMAGE_SIZE = 224
FEATURE_LAYER = 'layer3'
BACKGROUND_EROSION = 0
EPS = 1e-6

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
GROUNDING_MASK_ROOT = REPO_ROOT / 'external' / 'UniVAD' / 'masks' / 'mvtec_loco_caption'

PROBE_ROOT = EXP_ROOT / 'reports' / 'background_nuisance_probe'
FEATURE_RESPONSE_CSV = PROBE_ROOT / f'{CATEGORY}_background_foreground_shift_response.csv'
SUBSPACE_CSV = PROBE_ROOT / f'{CATEGORY}_background_shift_subspace.csv'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')
DRIVE_MASK_ROOT = Path('/content/drive/MyDrive/ReGraM/univad_masks/mvtec_loco_caption')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [
        parent / 'content' / dataset_name,
        parent / 'data' / 'row' / dataset_name,
    ]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    if (RAW_LOCO_ROOT / CATEGORY / 'test' / 'good').exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')


def expected_clean_mask_path(source_path: Path) -> Path:
    try:
        rel_path = source_path.relative_to(RAW_LOCO_ROOT / CATEGORY)
    except ValueError:
        parts = list(source_path.parts)
        category_index = len(parts) - 1 - parts[::-1].index(CATEGORY)
        rel_path = Path(*parts[category_index + 1 :])
    return GROUNDING_MASK_ROOT / CATEGORY / rel_path.with_suffix('') / 'grounding_mask.png'


def restore_clean_masks_from_drive_if_needed(source_paths: list[Path]) -> None:
    missing = [path for path in source_paths if not expected_clean_mask_path(path).exists()]
    if not missing:
        return
    mount_drive_if_available()
    copied = 0
    for source_path in missing:
        session_mask = expected_clean_mask_path(source_path)
        rel_path = session_mask.relative_to(GROUNDING_MASK_ROOT / CATEGORY)
        drive_mask = DRIVE_MASK_ROOT / CATEGORY / rel_path
        if not drive_mask.exists():
            continue
        session_mask.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_mask, session_mask)
        copied += 1
    print(f'restored clean masks from Drive: {copied}/{len(missing)}')


restore_raw_loco_from_drive_if_needed()

settings_df = pd.DataFrame([
    {
        'category': CATEGORY,
        'sample_ids': ', '.join(SAMPLE_IDS),
        'shift_specs': ', '.join(f'{name}/{severity}' for name, severity in SHIFT_SPECS),
        'raw_loco_root': str(RAW_LOCO_ROOT),
        'grounding_mask_root': str(GROUNDING_MASK_ROOT),
        'feature_layer': FEATURE_LAYER,
        'image_size': IMAGE_SIZE,
        'response_csv': str(FEATURE_RESPONSE_CSV),
    }
])
display(settings_df)


## Cell Role: Build Clean/Shifted Sample Table

manifest에서 clean source와 artificial shift parameter를 읽어 소수 샘플만 구성한다. 이 단계에서는 이미지를 저장하지 않고 메모리에서 augmentation을 적용한다.

In [ ]:
from data.augmentation_runtime import apply_augmentation, load_manifest
from PIL import Image


def resolve_source_path(entry: dict) -> Path:
    path = Path(entry['source_path'])
    if entry.get('source_path_mode') == 'repo_relative' or not path.is_absolute():
        return REPO_ROOT / path
    return path


def select_manifest_entries(augmentation_type: str, severity: str, sample_ids: list[str]) -> list[dict]:
    manifest_path = REPO_ROOT / 'manifests' / f'query_{augmentation_type}.jsonl'
    entries = [
        entry
        for entry in load_manifest(manifest_path)
        if entry.get('category') == CATEGORY
        and entry.get('augmentation_type') == augmentation_type
        and entry.get('severity') == severity
    ]
    by_source = {entry['source_id']: entry for entry in entries}
    selected = [by_source[source_id] for source_id in sample_ids if source_id in by_source]
    return selected if selected else entries[:len(sample_ids)]


probe_rows = []
source_paths_by_id: dict[str, Path] = {}
for augmentation_type, severity in SHIFT_SPECS:
    condition = f'{augmentation_type}_{severity}'
    for entry in select_manifest_entries(augmentation_type, severity, SAMPLE_IDS):
        source_path = resolve_source_path(entry)
        source_paths_by_id[entry['source_id']] = source_path
        probe_rows.append(
            {
                'base_image_id': entry['source_id'],
                'condition': condition,
                'augmentation_type': augmentation_type,
                'severity': severity,
                'seed': int(entry['seed']),
                'params': entry.get('params') or {},
                'source_path': str(source_path),
                'mask_path': str(expected_clean_mask_path(source_path)),
            }
        )

if not probe_rows:
    raise ValueError('No probe rows selected')

restore_clean_masks_from_drive_if_needed(list(source_paths_by_id.values()))
probe_df = pd.DataFrame(probe_rows)
display(probe_df)
missing_masks = [path for path in probe_df['mask_path'].map(Path) if not path.exists()]
if missing_masks:
    raise FileNotFoundError(f'Missing clean grounding masks: {missing_masks[:5]}')


## Cell Role: Build Frozen Backbone Feature Extractor

ImageNet-pretrained torchvision backbone의 intermediate feature map을 사용한다. 이 probe는 training-free이며 backbone parameter를 업데이트하지 않는다. 첫 실행 시 torchvision weight download가 필요할 수 있다.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

try:
    import torchvision
    from torchvision.models import ResNet18_Weights, resnet18
    from torchvision.models.feature_extraction import create_feature_extractor
    from torchvision.transforms import InterpolationMode
    from torchvision.transforms import functional as TF
except Exception as exc:
    raise ImportError('torchvision is required for this feature probe') from exc

weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights).to(device).eval()
feature_extractor = create_feature_extractor(backbone, return_nodes={FEATURE_LAYER: 'feat'}).to(device).eval()
mean = torch.tensor(weights.transforms().mean, dtype=torch.float32, device=device).view(1, 3, 1, 1)
std = torch.tensor(weights.transforms().std, dtype=torch.float32, device=device).view(1, 3, 1, 1)


def preprocess_image(image: Image.Image) -> torch.Tensor:
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BILINEAR)
    tensor = TF.to_tensor(image).unsqueeze(0).to(device)
    return (tensor - mean) / std


@torch.no_grad()
def extract_feature_map(image: Image.Image) -> torch.Tensor:
    outputs = feature_extractor(preprocess_image(image))
    return outputs['feat'].squeeze(0).detach().cpu()


def foreground_mask_from_grounding(mask_path: Path, feature_hw: tuple[int, int]) -> torch.Tensor:
    mask_rgb = np.asarray(Image.open(mask_path).convert('RGB'))
    foreground = np.any(mask_rgb != 0, axis=-1).astype(np.float32)
    fg = torch.from_numpy(foreground)[None, None]
    fg = F.interpolate(fg, size=feature_hw, mode='nearest').squeeze().bool()
    if BACKGROUND_EROSION > 0:
        # Keep the default at 0. Erosion is intentionally omitted unless a future probe needs it.
        pass
    return fg


def masked_mean_std(feature_map: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, int]:
    flattened = feature_map.flatten(1).T
    mask_flat = mask.flatten()
    selected = flattened[mask_flat]
    if selected.numel() == 0:
        selected = flattened
    return selected.mean(dim=0), selected.std(dim=0, unbiased=False).clamp_min(EPS), int(mask_flat.sum().item())

print('backbone = torchvision.resnet18', 'feature_layer =', FEATURE_LAYER)


## Cell Role: Background/Foreground Shift Response Metrics

같은 clean image와 shifted image의 feature map 차이를 background/foreground에서 각각 계산한다. 핵심은 background Δ가 foreground Δ와 align되는지, 그리고 background mean shift나 moment matching이 foreground clean distance를 줄이는지다.

In [ ]:
def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    return float(torch.dot(a, b) / (a.norm() * b.norm()).clamp_min(EPS))


def relative_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    return float((a - b).norm())


def moment_match_vector(
    x: torch.Tensor,
    *,
    source_mean: torch.Tensor,
    source_std: torch.Tensor,
    target_mean: torch.Tensor,
    target_std: torch.Tensor,
) -> torch.Tensor:
    return ((x - target_mean) / target_std.clamp_min(EPS)) * source_std + source_mean


response_rows = []
bg_delta_vectors = []
fg_delta_vectors = []
row_keys = []
feature_shape = None

for _, row in probe_df.iterrows():
    source_path = Path(row['source_path'])
    clean_image = Image.open(source_path).convert('RGB')
    shifted_image = apply_augmentation(
        clean_image,
        augmentation_type=row['augmentation_type'],
        severity=row['severity'],
        seed=int(row['seed']),
        params=row['params'],
    )

    clean_feat = extract_feature_map(clean_image)
    shifted_feat = extract_feature_map(shifted_image)
    _, feat_h, feat_w = clean_feat.shape
    feature_shape = (int(feat_h), int(feat_w))
    fg_mask = foreground_mask_from_grounding(Path(row['mask_path']), (feat_h, feat_w))
    bg_mask = ~fg_mask

    clean_bg_mean, clean_bg_std, bg_pixels = masked_mean_std(clean_feat, bg_mask)
    shifted_bg_mean, shifted_bg_std, _ = masked_mean_std(shifted_feat, bg_mask)
    clean_fg_mean, clean_fg_std, fg_pixels = masked_mean_std(clean_feat, fg_mask)
    shifted_fg_mean, shifted_fg_std, _ = masked_mean_std(shifted_feat, fg_mask)

    bg_delta = shifted_bg_mean - clean_bg_mean
    fg_delta = shifted_fg_mean - clean_fg_mean
    bg_mean_corrected_fg = shifted_fg_mean - bg_delta
    moment_corrected_fg = moment_match_vector(
        shifted_fg_mean,
        source_mean=clean_bg_mean,
        source_std=clean_bg_std,
        target_mean=shifted_bg_mean,
        target_std=shifted_bg_std,
    )

    before_dist = relative_distance(shifted_fg_mean, clean_fg_mean)
    bg_corrected_dist = relative_distance(bg_mean_corrected_fg, clean_fg_mean)
    moment_corrected_dist = relative_distance(moment_corrected_fg, clean_fg_mean)

    bg_delta_vectors.append(bg_delta)
    fg_delta_vectors.append(fg_delta)
    row_keys.append((row['base_image_id'], row['condition']))
    response_rows.append(
        {
            'base_image_id': row['base_image_id'],
            'condition': row['condition'],
            'augmentation_type': row['augmentation_type'],
            'severity': row['severity'],
            'bg_feature_cells': bg_pixels,
            'fg_feature_cells': fg_pixels,
            'bg_delta_norm': float(bg_delta.norm()),
            'fg_delta_norm': float(fg_delta.norm()),
            'bg_fg_delta_cosine': cosine(bg_delta, fg_delta),
            'fg_clean_shifted_distance': before_dist,
            'fg_distance_after_bg_mean_correction': bg_corrected_dist,
            'fg_distance_after_bg_moment_matching': moment_corrected_dist,
            'bg_mean_correction_improvement': before_dist - bg_corrected_dist,
            'bg_moment_matching_improvement': before_dist - moment_corrected_dist,
            'bg_mean_correction_ratio': bg_corrected_dist / max(before_dist, EPS),
            'bg_moment_matching_ratio': moment_corrected_dist / max(before_dist, EPS),
            'feature_layer': FEATURE_LAYER,
            'feature_shape': str(feature_shape),
        }
    )

response_df = pd.DataFrame(response_rows)
response_df.to_csv(FEATURE_RESPONSE_CSV, index=False)
print(f'saved response metrics: {FEATURE_RESPONSE_CSV}')
display(response_df)
display(response_df.groupby('condition')[[
    'bg_fg_delta_cosine',
    'fg_clean_shifted_distance',
    'fg_distance_after_bg_mean_correction',
    'fg_distance_after_bg_moment_matching',
    'bg_mean_correction_improvement',
    'bg_moment_matching_improvement',
]].agg(['mean', 'median', 'min', 'max']))


## Cell Role: Background Shift-Response Subspace

background Δ vector들로 SVD를 수행해 nuisance subspace가 low-rank인지 확인한다. 이후 foreground Δ가 이 subspace에 얼마나 투영되는지 본다. 여기서 높은 explained variance와 높은 foreground projection ratio가 나오면 shared nuisance component 가정이 더 방어 가능해진다.

In [ ]:
bg_delta_matrix = torch.stack(bg_delta_vectors, dim=0)
fg_delta_matrix = torch.stack(fg_delta_vectors, dim=0)
centered_bg = bg_delta_matrix - bg_delta_matrix.mean(dim=0, keepdim=True)

if centered_bg.shape[0] < 2:
    raise ValueError('Need at least two background delta samples for SVD')

U, S, Vh = torch.linalg.svd(centered_bg, full_matrices=False)
variance = S**2
explained = variance / variance.sum().clamp_min(EPS)
cumulative = torch.cumsum(explained, dim=0)

subspace_rows = []
max_k = min(8, Vh.shape[0])
for k in range(1, max_k + 1):
    basis = Vh[:k].T
    for index, fg_delta in enumerate(fg_delta_matrix):
        projection = basis @ (basis.T @ fg_delta)
        projection_ratio = float(projection.norm() / fg_delta.norm().clamp_min(EPS))
        residual_ratio = float((fg_delta - projection).norm() / fg_delta.norm().clamp_min(EPS))
        base_image_id, condition = row_keys[index]
        subspace_rows.append(
            {
                'k': k,
                'base_image_id': base_image_id,
                'condition': condition,
                'bg_explained_variance_cumulative': float(cumulative[k - 1]),
                'fg_projection_ratio': projection_ratio,
                'fg_residual_ratio_after_removal': residual_ratio,
            }
        )

subspace_df = pd.DataFrame(subspace_rows)
subspace_df.to_csv(SUBSPACE_CSV, index=False)
print(f'saved subspace metrics: {SUBSPACE_CSV}')
print('SVD explained variance:', [round(float(value), 4) for value in explained[:max_k]])
display(subspace_df)
display(subspace_df.groupby(['condition', 'k'])[[
    'bg_explained_variance_cumulative',
    'fg_projection_ratio',
    'fg_residual_ratio_after_removal',
]].mean().reset_index())


## Cell Role: Visualize Probe Results

condition별 background/foreground alignment와 correction 효과를 본다. conservative하게 해석해야 한다. background correction ratio가 1보다 낮아야 foreground clean distance가 줄어든 것이다.

In [ ]:
import matplotlib.pyplot as plt

if response_df.empty:
    print('No response rows to plot')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    response_df.boxplot(column='bg_fg_delta_cosine', by='condition', ax=axes[0], rot=25)
    axes[0].set_title('background vs foreground delta cosine')
    axes[0].set_ylabel('cosine')

    ratio_df = response_df[['condition', 'bg_mean_correction_ratio', 'bg_moment_matching_ratio']].copy()
    ratio_df.groupby('condition').mean().plot(kind='bar', ax=axes[1])
    axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1)
    axes[1].set_title('foreground distance ratio after correction')
    axes[1].set_ylabel('after / before')
    axes[1].tick_params(axis='x', rotation=25)

    response_df.groupby('condition')[['bg_delta_norm', 'fg_delta_norm']].mean().plot(kind='bar', ax=axes[2])
    axes[2].set_title('mean shift response norm')
    axes[2].tick_params(axis='x', rotation=25)

    plt.suptitle('')
    plt.tight_layout()
    plt.show()

if not subspace_df.empty:
    selected_k = min(3, int(subspace_df['k'].max()))
    plot_df = subspace_df[subspace_df['k'] == selected_k]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_df.boxplot(column='fg_projection_ratio', by='condition', ax=axes[0], rot=25)
    axes[0].set_title(f'foreground delta projection ratio, k={selected_k}')
    axes[0].set_ylabel('projection / fg delta norm')

    spectrum = subspace_df[['k', 'bg_explained_variance_cumulative']].drop_duplicates().sort_values('k')
    axes[1].plot(spectrum['k'], spectrum['bg_explained_variance_cumulative'], marker='o')
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title('background delta cumulative explained variance')
    axes[1].set_xlabel('k')
    axes[1].set_ylabel('cumulative variance')

    plt.suptitle('')
    plt.tight_layout()
    plt.show()


## Cell Role: Interpretation Checklist

다음 조건이 동시에 보여야 다음 단계로 갈 수 있다.

- `bg_fg_delta_cosine`이 shift condition별로 양수이고 충분히 안정적이다.
- background mean correction 또는 moment matching 후 foreground clean distance ratio가 1보다 낮다.
- background Δ SVD spectrum이 어느 정도 low-rank다.
- foreground Δ가 background nuisance subspace에 의미 있게 projection된다.

이게 안 나오면 background-derived calibration은 method가 아니라 실패한 가정으로 정리하는 것이 맞다.